# Zip the complete `dementia_progress_v0` project folder

This notebook creates a timestamped zip archive of the full project folder located at `../../dementia_progress_v0`.

The archive is written outside the source project folder to prevent the zip file from being recursively included inside itself. The zip preserves the top-level project folder name so that extracting it recreates `dementia_progress_v0/` with all files underneath it.

In [1]:
from pathlib import Path
from datetime import datetime
import os
import zipfile

# ---------------------------------------------------------------------
# User-controlled paths
# ---------------------------------------------------------------------
# Run this notebook from the notebooks/ directory, where this relative
# path resolves to the full project folder.
PROJECT_DIR = Path("../../dementia_progress_v0").expanduser().resolve()

# Keep the archive outside PROJECT_DIR to avoid recursive zip inclusion.
OUTPUT_DIR = PROJECT_DIR.parent / "project_archives"
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
ZIP_PATH = OUTPUT_DIR / f"{PROJECT_DIR.name}_complete_project_{TIMESTAMP}.zip"

# Optional exclusions. Leave empty for a complete project archive.
EXCLUDE_DIR_NAMES = set()
EXCLUDE_FILE_NAMES = set()

print(f"Project directory: {PROJECT_DIR}")
print(f"Archive output:    {ZIP_PATH}")

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"PROJECT_DIR does not exist: {PROJECT_DIR}\n"
        "Check that the notebook is being run from the notebooks/ directory, "
        "or edit PROJECT_DIR above."
    )

if not PROJECT_DIR.is_dir():
    raise NotADirectoryError(f"PROJECT_DIR is not a directory: {PROJECT_DIR}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Project directory: /home/ppanta/puru_proj/1_dementia_progress/dementia_progress_v0
Archive output:    /home/ppanta/puru_proj/1_dementia_progress/project_archives/dementia_progress_v0_complete_project_20260616_200155.zip


## Count files before zipping

In [2]:
def should_exclude(path: Path) -> bool:
    """Return True only for explicitly excluded files/directories."""
    parts = set(path.parts)
    if parts.intersection(EXCLUDE_DIR_NAMES):
        return True
    if path.name in EXCLUDE_FILE_NAMES:
        return True
    return False

all_files = [
    p for p in PROJECT_DIR.rglob("*")
    if p.is_file() and not should_exclude(p)
]

total_bytes = sum(p.stat().st_size for p in all_files)
print(f"Files to archive: {len(all_files):,}")
print(f"Uncompressed size: {total_bytes / (1024**2):,.2f} MB")

Files to archive: 45,542
Uncompressed size: 8,235.68 MB


## Create the zip archive

In [3]:
def zip_project_folder(project_dir: Path, zip_path: Path) -> dict:
    """Zip the complete project folder and return archive statistics."""
    if zip_path.exists():
        zip_path.unlink()

    project_parent = project_dir.parent
    n_files = 0
    n_bytes = 0

    with zipfile.ZipFile(zip_path, mode="w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for path in sorted(project_dir.rglob("*")):
            if should_exclude(path):
                continue
            if path.is_file():
                # Preserve the top-level project folder in the archive.
                arcname = path.relative_to(project_parent)
                zf.write(path, arcname=arcname)
                n_files += 1
                n_bytes += path.stat().st_size

    return {
        "zip_path": zip_path,
        "files_archived": n_files,
        "uncompressed_mb": n_bytes / (1024**2),
        "zip_size_mb": zip_path.stat().st_size / (1024**2),
    }

stats = zip_project_folder(PROJECT_DIR, ZIP_PATH)

print("Archive created successfully.")
print(f"Zip file:           {stats['zip_path']}")
print(f"Files archived:     {stats['files_archived']:,}")
print(f"Uncompressed size:  {stats['uncompressed_mb']:,.2f} MB")
print(f"Compressed zip size:{stats['zip_size_mb']:,.2f} MB")

Archive created successfully.
Zip file:           /home/ppanta/puru_proj/1_dementia_progress/project_archives/dementia_progress_v0_complete_project_20260616_200155.zip
Files archived:     45,542
Uncompressed size:  8,235.68 MB
Compressed zip size:3,973.68 MB


## Integrity check

In [4]:
with zipfile.ZipFile(ZIP_PATH, mode="r") as zf:
    bad_file = zf.testzip()
    archived_names = zf.namelist()

if bad_file is None:
    print("Zip integrity check passed: no corrupted files detected.")
else:
    raise RuntimeError(f"Zip integrity check failed at: {bad_file}")

print(f"Archived entries: {len(archived_names):,}")
print("First 10 archived entries:")
for name in archived_names[:10]:
    print(f"  {name}")

Zip integrity check passed: no corrupted files detected.
Archived entries: 45,542
First 10 archived entries:
  dementia_progress_v0/.gitignore
  dementia_progress_v0/.ipynb_checkpoints/requirements-checkpoint.txt
  dementia_progress_v0/CACHEDIR.TAG
  dementia_progress_v0/VERSION.txt
  dementia_progress_v0/arch/op_full_archive_20260529_132056.zip
  dementia_progress_v0/bin/activate
  dementia_progress_v0/bin/activate.csh
  dementia_progress_v0/bin/activate.fish
  dementia_progress_v0/bin/activate.nu
  dementia_progress_v0/bin/activate.ps1


## Optional extraction test

Run this cell only when you want to verify extraction into a temporary directory. It does not modify the original project folder.